# Historical experiment: OneTransformer(1).ipynb
Archived source with outputs/execution counts cleared. Superseded by the numbered notebooks.
Original paths and experimental branches are retained for reference; this is not the supported run sequence.

In [ ]:
import torch

from transformers import AutoModelForCausalLM, AutoTokenizer

In [ ]:
def load_tinystories_model(model_id, revision, attn_implementation="eager"):
    tokenizer = AutoTokenizer.from_pretrained(model_id, revision=revision)

    model = AutoModelForCausalLM.from_pretrained(model_id, revision=revision, attn_implementation=attn_implementation)

    model.eval()

    return model, tokenizer

In [ ]:

model_id = "roneneldan/TinyStories-3M"
revision = "cfaf26ec85ecdfc1bd7c2638104cce55cb67f894"

model, tokenizer = load_tinystories_model(model_id, revision)

In [ ]:
print("Model class:", type(model).__name__)
print("Number of blocks:", len(model.transformer.h))
print("Hidden dimension:", model.config.hidden_size)
print("Attention heads:", model.config.num_heads)
print("Vocabulary size:", model.config.vocab_size)
print("Training mode:", model.training)

In [ ]:
print(type(tokenizer))

In [ ]:
PROMPT_LENGTH = 32
GENERATED_LENGTH = 20

prompt = (
    "Once upon a time, a curious little girl named Lily lived in a "
    "quiet village beside a green forest. Every morning, she carried "
    "fresh bread to her grandmother and watched the birds in the trees."
)

encoded = tokenizer(
    prompt,
    add_special_tokens=False,
    return_tensors="pt",
)

available_tokens = encoded["input_ids"].shape[1]

print("Original prompt token count:", available_tokens)

In [ ]:
if available_tokens < PROMPT_LENGTH:
    raise ValueError(
        f"Prompt contains only {available_tokens} tokens. "
        f"At least {PROMPT_LENGTH} are required."
    )

input_ids = encoded["input_ids"][:, :PROMPT_LENGTH]
attention_mask = torch.ones_like(input_ids)

print("Benchmark input shape:", input_ids.shape)
print("Benchmark token count:", input_ids.shape[1])
print("Token IDs:", input_ids[0].tolist())
print("Truncated prompt:", tokenizer.decode(input_ids[0]))

assert input_ids.shape == (1, 32)

In [ ]:
block_input_lengths = []

def record_block_input_length(module, args):
    hidden_states = args[0]

    block_input_lengths.append(
        hidden_states.shape[1]
    )

hook_handle = (
    model.transformer.h[0]
    .register_forward_pre_hook(
        record_block_input_length
    )
)

In [ ]:
with torch.no_grad():
    generated_sequences = model.generate(
        input_ids=input_ids,
        attention_mask=attention_mask,

        min_new_tokens=GENERATED_LENGTH,
        max_new_tokens=GENERATED_LENGTH,

        do_sample=False,
        use_cache=True,

        pad_token_id=tokenizer.eos_token_id,
    )

hook_handle.remove()

In [ ]:
generated_token_ids = generated_sequences[
    :,
    PROMPT_LENGTH:
]

print("Prompt tokens:", input_ids.shape[1])
print("Generated tokens:", generated_token_ids.shape[1])
print("Total tokens:", generated_sequences.shape[1])

assert input_ids.shape[1] == 32
assert generated_token_ids.shape[1] == 20
assert generated_sequences.shape[1] == 52

In [ ]:
print(
    "Generated continuation:",
    tokenizer.decode(
        generated_token_ids[0],
        skip_special_tokens=True,
    ),
)

print(
    "Complete sequence:",
    tokenizer.decode(
        generated_sequences[0],
        skip_special_tokens=True,
    ),
)

In [ ]:
assert block_input_lengths[0] == 32
assert len(block_input_lengths) == 20
assert all(
    length == 1
    for length in block_input_lengths[1:]
)

print("KV-cache behavior verified")

In [ ]:
def create_block0_input(model, input_ids):
    batch_size, sequence_length = input_ids.shape

    position_ids = torch.arange(
        sequence_length,
        dtype=torch.long,
        device=input_ids.device,
    ).unsqueeze(0)

    with torch.no_grad():
        token_embeddings = model.transformer.wte(
            input_ids
        )

        position_embeddings = model.transformer.wpe(
            position_ids
        )

        block_input = model.transformer.drop(
            token_embeddings + position_embeddings
        )

    return block_input

In [ ]:
block_input = create_block0_input(
    model,
    input_ids,
)

In [ ]:
block_input.shape

In [ ]:
print("Token ID shape:", input_ids.shape)
print("Block input shape:", block_input.shape)
print("Block input dtype:", block_input.dtype)

print(
    "Block input range:",
    block_input.min().item(),
    block_input.max().item(),
)

In [ ]:
from torch import nn

In [ ]:
class BlockWrapper(nn.Module):
    def __init__(self, transformer_block):
        super().__init__()

        self.transformer_block = transformer_block

    def forward(self, hidden_states):
        block_outputs = self.transformer_block(
            hidden_states,
            layer_past=None,
            attention_mask=None,
            use_cache=False,
            output_attentions=False,
        )

        output_hidden_states = block_outputs[0]

        return output_hidden_states

In [ ]:
block0 = BlockWrapper(
    model.transformer.h[0]
)

block0.eval()

In [ ]:
with torch.no_grad():
    block_output_fp32 = block0(
        block_input
    )

In [ ]:
print("Block input shape:",
      block_input.shape)

print("Block output shape:",
      block_output_fp32.shape)

print("Input dtype:",
      block_input.dtype)

print("Output dtype:",
      block_output_fp32.dtype)

print(
    "Output range:",
    block_output_fp32.min().item(),
    block_output_fp32.max().item(),
)

In [ ]:
def export_block(
    block,
    example_input,
):
    block.eval()

    example_arguments = (
        example_input.detach(),
    )
    print(type(example_input))
    print(type(example_arguments))

    exported_program = torch.export.export(
        block,
        example_arguments,
    )

    return exported_program

In [ ]:
exported_block0 = export_block(
    block0,
    block_input,
)

In [ ]:
print(type(exported_block0))
print(
    "Graph node count:",
    len(list(exported_block0.graph.nodes)),
)

In [ ]:
exported_block0_module = (
    exported_block0.module()
)

In [ ]:
with torch.no_grad():
    exported_output = (
        exported_block0_module(
            block_input
        )
    )

In [ ]:
maximum_difference = (
    block_output_fp32
    - exported_output
).abs().max().item()

print(
    "Maximum original/exported difference:",
    maximum_difference,
)

print(
    "Exported output shape:",
    exported_output.shape,
)

In [ ]:
from litert_torch.quantize.pt2e_quantizer import (
    PT2EQuantizer,
    get_symmetric_quantization_config,
)

def create_w8a8_quantizer():
    quantization_config = get_symmetric_quantization_config(
        is_per_channel=False,
        is_dynamic=False,
        is_qat=False,
    )

    quantizer = PT2EQuantizer().set_global(
        quantization_config
    )

    return quantizer


quantizer = create_w8a8_quantizer()
print(quantizer)

In [ ]:
from torchao.quantization.pt2e.quantize_pt2e import prepare_pt2e

prepared_block0 = prepare_pt2e(
    exported_block0.module(),
    quantizer,
)

print(prepared_block0)

In [ ]:
def calibrate_block(prepared_block, calibration_input):
    with torch.no_grad():
        output = prepared_block(calibration_input)

    return output


calibration_output = calibrate_block(
    prepared_block0,
    block_input,
)

In [ ]:
observer = prepared_block0.activation_post_process_0

print("Minimum:", observer.min_val)
print("Maximum:", observer.max_val)
print("Output shape:", calibration_output.shape)

In [ ]:
from torchao.quantization.pt2e.quantize_pt2e import convert_pt2e

quantized_block0 = convert_pt2e(prepared_block0)

print(quantized_block0)

In [ ]:
with torch.no_grad():
    quantized_output = quantized_block0(block_input)

print("Output shape:", quantized_output.shape)
print("Output dtype:", quantized_output.dtype)

In [ ]:
absolute_error = (
    block_output_fp32 - quantized_output
).abs()

print("Maximum absolute error:", absolute_error.max().item())
print("Mean absolute error:", absolute_error.mean().item())

In [ ]:
def compare_outputs(test_block_input):
    with torch.no_grad():
        fp32_output = block0(test_block_input)
        int8_output = quantized_block0(test_block_input)

    error = (fp32_output - int8_output).abs()

    print("Maximum absolute error:",
          error.max().item())

    print("Mean absolute error:",
          error.mean().item())

In [ ]:
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

In [ ]:
new_prompt = "The little dog walked into the forest"

encoded = tokenizer(
    new_prompt,
    return_tensors="pt",
    padding="max_length",
    truncation=True,
    max_length=32,
)

new_input_ids = encoded["input_ids"]
attention_mask = encoded["attention_mask"]

print("Input shape:", new_input_ids.shape)
print("Real token count:", attention_mask.sum().item())
print("Pad token:", tokenizer.pad_token)
print("Pad token ID:", tokenizer.pad_token_id)

In [ ]:
sequence_length = new_input_ids.shape[1]

new_position_ids = torch.arange(
    sequence_length,
    device=new_input_ids.device,
).unsqueeze(0)

with torch.no_grad():
    new_block_input = (
        model.transformer.wte(new_input_ids)
        + model.transformer.wpe(new_position_ids)
    )

In [ ]:
with torch.no_grad():
    fp32_output = block0(new_block_input)
    quantized_output = quantized_block0(new_block_input)

absolute_error = (
    fp32_output - quantized_output
).abs()

print("Maximum absolute error:",
      absolute_error.max().item())

print("Mean absolute error:",
      absolute_error.mean().item())

In [ ]:
real_token_mask = (
    attention_mask
    .unsqueeze(-1)
    .expand_as(absolute_error)
    .bool()
)

real_token_error = absolute_error[real_token_mask]

print(
    "Real-token maximum error:",
    real_token_error.max().item(),
)

print(
    "Real-token mean error:",
    real_token_error.mean().item(),
)

In [ ]:
fresh_exported_block0 = export_block(
    block0,
    block_input,
)

fresh_quantizer = create_w8a8_quantizer()

prepared_block0_multi = prepare_pt2e(
    fresh_exported_block0.module(),
    fresh_quantizer,
)

In [ ]:
calibration_prompts = [
    """
    Once upon a time, a little girl named Lily lived beside a quiet
    forest. Every morning she carried bread to her grandmother and
    greeted all the animals along the path.
    """,

    """
    Tom found a small red ball underneath his bed. He took it outside
    and played with his dog until the sun disappeared behind the hills.
    """,

    """
    A young rabbit wanted to cross the river, but the water was too
    deep. His friends gathered branches and helped him build a bridge.
    """,

    """
    Mia planted a tiny seed in the garden. She watered it every day,
    and soon a tall sunflower grew beside her bedroom window.
    """,

    """
    The little bird could not fly as high as the others. She practiced
    every afternoon until she finally reached the top of the old tree.
    """,

    """
    Ben heard a strange sound inside the kitchen. He opened the door
    slowly and discovered a hungry kitten looking for some food.
    """,

    """
    A kind farmer shared his vegetables with the people in his village.
    When winter arrived, everyone helped him repair his wooden barn.
    """,

    """
    Sara lost her favorite blue hat at the park. She searched near the
    swings and found it beside a sleeping brown puppy.
    """,
]

In [ ]:
all_calibration_ids = []

for prompt in calibration_prompts:
    token_ids = tokenizer.encode(
        prompt,
        add_special_tokens=False,
    )

    all_calibration_ids.extend(token_ids)
    all_calibration_ids.append(tokenizer.eos_token_id)


calibration_chunks = []

for start in range(
    0,
    len(all_calibration_ids) - 31,
    32,
):
    chunk = all_calibration_ids[
        start : start + 32
    ]

    calibration_chunks.append(chunk)


print("Number of calibration chunks:",
      len(calibration_chunks))

print("Tokens in each chunk:",
      len(calibration_chunks[0]))

In [ ]:
position_ids_32 = torch.arange(
    32,
).unsqueeze(0)

with torch.no_grad():
    for chunk in calibration_chunks:
        calibration_ids = torch.tensor(
            [chunk],
            dtype=torch.long,
        )

        calibration_block_input = (
            model.transformer.wte(calibration_ids)
            + model.transformer.wpe(position_ids_32)
        )

        prepared_block0_multi(
            calibration_block_input
        )

In [ ]:
observer = (
    prepared_block0_multi
    .activation_post_process_0
)

print("Observed minimum:", observer.min_val)
print("Observed maximum:", observer.max_val)

In [ ]:
quantized_block0_multi = convert_pt2e(
    prepared_block0_multi
)

In [ ]:
with torch.no_grad():
    fp32_test_output = block0(
        new_block_input
    )

    quantized_test_output = (
        quantized_block0_multi(
            new_block_input
        )
    )

In [ ]:
multi_calibration_error = (
    fp32_test_output
    - quantized_test_output
).abs()

print(
    "Output shape:",
    quantized_test_output.shape,
)

print(
    "Output dtype:",
    quantized_test_output.dtype,
)

print(
    "Maximum absolute error:",
    multi_calibration_error.max().item(),
)

print(
    "Mean absolute error:",
    multi_calibration_error.mean().item(),
)

In [ ]:
real_token_mask = (
    attention_mask
    .unsqueeze(-1)
    .expand_as(multi_calibration_error)
    .bool()
)

real_token_error_multi = (
    multi_calibration_error[
        real_token_mask
    ]
)

print(
    "Real-token maximum error:",
    real_token_error_multi.max().item(),
)

print(
    "Real-token mean error:",
    real_token_error_multi.mean().item(),
)

In [ ]:
litert_exported_block0 = export_block(
    block0,
    block_input,
)

litert_quantizer = create_w8a8_quantizer()

litert_prepared_block0 = prepare_pt2e(
    litert_exported_block0.module(),
    litert_quantizer,
)

position_ids_32 = torch.arange(32).unsqueeze(0)

with torch.no_grad():
    for chunk in calibration_chunks:
        calibration_ids = torch.tensor(
            [chunk],
            dtype=torch.long,
        )

        calibration_block_input = (
            model.transformer.wte(calibration_ids)
            + model.transformer.wpe(position_ids_32)
        )

        litert_prepared_block0(
            calibration_block_input
        )

In [ ]:
litert_quantized_block0 = convert_pt2e(
    litert_prepared_block0,
    fold_quantize=False,
)

In [ ]:
import litert_torch

from litert_torch.quantize.quant_config import (
    QuantConfig,
)

sample_args = (
    block_input.detach(),
)

edge_block0 = litert_torch.convert(
    litert_quantized_block0,
    sample_args,
    quant_config=QuantConfig(
        pt2e_quantizer=litert_quantizer
    ),
)

In [ ]:
import numpy as np
import torch

with torch.no_grad():
    fp32_output = block0(
        new_block_input
    )

    pt2e_output = litert_quantized_block0(
        new_block_input
    )

litert_output = edge_block0(
    new_block_input.detach()
)

In [ ]:
if isinstance(litert_output, (list, tuple)):
    litert_output = litert_output[0]

fp32_numpy = (
    fp32_output.detach().cpu().numpy()
)

pt2e_numpy = (
    pt2e_output.detach().cpu().numpy()
)

litert_numpy = np.asarray(litert_output)

print("FP32 shape:", fp32_numpy.shape)
print("PT2E shape:", pt2e_numpy.shape)
print("LiteRT shape:", litert_numpy.shape)
print("LiteRT dtype:", litert_numpy.dtype)

In [ ]:
conversion_error = np.abs(
    pt2e_numpy - litert_numpy
)

print(
    "PT2E/LiteRT maximum error:",
    conversion_error.max(),
)

print(
    "PT2E/LiteRT mean error:",
    conversion_error.mean(),
)

print(
    "PT2E/LiteRT close:",
    np.allclose(
        pt2e_numpy,
        litert_numpy,
        atol=1e-4,
        rtol=1e-4,
    ),
)

In [ ]:
fp32_litert_error = np.abs(
    fp32_numpy - litert_numpy
)

print(
    "FP32/LiteRT maximum error:",
    fp32_litert_error.max(),
)

print(
    "FP32/LiteRT mean error:",
    fp32_litert_error.mean(),
)

In [ ]:
real_token_mask_numpy = np.broadcast_to(
    attention_mask
    .cpu()
    .numpy()
    .astype(bool)[..., None],
    fp32_litert_error.shape,
)

real_token_litert_error = (
    fp32_litert_error[
        real_token_mask_numpy
    ]
)

print(
    "Real-token FP32/LiteRT maximum:",
    real_token_litert_error.max(),
)

print(
    "Real-token FP32/LiteRT mean:",
    real_token_litert_error.mean(),
)

In [ ]:
import os

tflite_path = "block0_w8a8.tflite"

edge_block0.export(tflite_path)

print(
    "File exists:",
    os.path.isfile(tflite_path),
)

print(
    "File size:",
    os.path.getsize(tflite_path),
    "bytes",
)

In [ ]:
reloaded_block0 = litert_torch.load(
    tflite_path
)

reloaded_output = reloaded_block0(
    new_block_input.detach()
)

if isinstance(reloaded_output, (list, tuple)):
    reloaded_output = reloaded_output[0]

reloaded_numpy = np.asarray(
    reloaded_output
)

reload_error = np.abs(
    litert_numpy - reloaded_numpy
)

print(
    "Original/reloaded maximum error:",
    reload_error.max(),
)

print(
    "Original/reloaded mean error:",
    reload_error.mean(),
)

In [ ]:
from ai_edge_litert.interpreter import Interpreter
import numpy as np
from collections import Counter

interpreter = Interpreter(
    model_path="block0_w8a8.tflite",
    experimental_preserve_all_tensors=True,
)

interpreter.allocate_tensors()

tensor_details = interpreter.get_tensor_details()

tensor_by_index = {
    tensor["index"]: tensor
    for tensor in tensor_details
}

In [ ]:
dtype_counts = Counter(
    np.dtype(tensor["dtype"]).name
    for tensor in tensor_details
)

print("Tensor dtype counts:")

for dtype, count in dtype_counts.items():
    print(dtype, count)

In [ ]:
matrix_operator_names = {
    "FULLY_CONNECTED",
    "BATCH_MATMUL",
}

w8a8_operator_count = 0

for operator_number, operator in enumerate(
    interpreter._get_ops_details()
):
    operator_name = operator["op_name"]

    if operator_name not in matrix_operator_names:
        continue

    input_tensors = [
        tensor_by_index[int(index)]
        for index in operator["inputs"]
        if int(index) >= 0
    ]

    output_tensors = [
        tensor_by_index[int(index)]
        for index in operator["outputs"]
        if int(index) >= 0
    ]

    print(
        f"\nOperator {operator_number}:",
        operator_name,
    )

    for position, tensor in enumerate(input_tensors):
        qparams = tensor["quantization_parameters"]

        print(
            f"  Input {position}:",
            "dtype =",
            np.dtype(tensor["dtype"]).name,
            "shape =",
            tensor["shape"].tolist(),
            "scale count =",
            len(qparams["scales"]),
            "name =",
            tensor["name"],
        )

    for position, tensor in enumerate(output_tensors):
        qparams = tensor["quantization_parameters"]

        print(
            f"  Output {position}:",
            "dtype =",
            np.dtype(tensor["dtype"]).name,
            "shape =",
            tensor["shape"].tolist(),
            "scale count =",
            len(qparams["scales"]),
        )

    if (
        len(input_tensors) >= 2
        and input_tensors[0]["dtype"] == np.int8
        and input_tensors[1]["dtype"] == np.int8
    ):
        w8a8_operator_count += 1
        print("  Classification: W8A8")
    else:
        print("  Classification: not W8A8")

print(
    "\nNumber of W8A8 matrix operators:",
    w8a8_operator_count,
)

In [ ]:
for tensor in tensor_details:
    if tensor["dtype"] == np.float32:
        print(
            "shape =",
            tensor["shape"].tolist(),
            "name =",
            tensor["name"],
        )

In [ ]:
import numpy as np

matrix_operations = {
    "FULLY_CONNECTED",
    "BATCH_MATMUL",
}

matrix_number = 0

for operator_index, operator in enumerate(
    interpreter._get_ops_details()
):
    if operator["op_name"] not in matrix_operations:
        continue

    matrix_number += 1

    print(
        "\n================================"
    )
    print("Matrix operation:", matrix_number)
    print("Graph operator index:", operator_index)
    print("Operator:", operator["op_name"])

    print("\nInputs:")

    for input_position, tensor_index in enumerate(
        operator["inputs"]
    ):
        tensor_index = int(tensor_index)

        if tensor_index < 0:
            continue

        tensor = tensor_by_index[tensor_index]
        qparams = tensor["quantization_parameters"]

        print(
            f"  Input {input_position}"
        )
        print("    tensor index:",
              tensor_index)
        print("    name:",
              tensor["name"])
        print("    shape:",
              tensor["shape"].tolist())
        print("    dtype:",
              np.dtype(tensor["dtype"]).name)
        print("    scales:",
              qparams["scales"])
        print("    zero points:",
              qparams["zero_points"])

    print("\nOutputs:")

    for output_position, tensor_index in enumerate(
        operator["outputs"]
    ):
        tensor_index = int(tensor_index)
        tensor = tensor_by_index[tensor_index]
        qparams = tensor["quantization_parameters"]

        print(
            f"  Output {output_position}"
        )
        print("    tensor index:",
              tensor_index)
        print("    name:",
              tensor["name"])
        print("    shape:",
              tensor["shape"].tolist())
        print("    dtype:",
              np.dtype(tensor["dtype"]).name)
        print("    scales:",
              qparams["scales"])
        print("    zero points:",
              qparams["zero_points"])

In [ ]:
import numpy as np

model_input = interpreter.get_input_details()[0]

test_input_numpy = (
    new_block_input
    .detach()
    .cpu()
    .numpy()
    .astype(np.float32)
)

interpreter.set_tensor(
    model_input["index"],
    test_input_numpy,
)

interpreter.invoke()

In [ ]:
q_proj_input_int8 = (
    interpreter.get_tensor(45).copy()
)

q_proj_weight_int8 = (
    interpreter.get_tensor(27).copy()
)

q_proj_output_int8 = (
    interpreter.get_tensor(46).copy()
)

In [ ]:
print(
    "Input:",
    q_proj_input_int8.shape,
    q_proj_input_int8.dtype,
    q_proj_input_int8.min(),
    q_proj_input_int8.max(),
)

print(
    "Weight:",
    q_proj_weight_int8.shape,
    q_proj_weight_int8.dtype,
    q_proj_weight_int8.min(),
    q_proj_weight_int8.max(),
)

print(
    "Output:",
    q_proj_output_int8.shape,
    q_proj_output_int8.dtype,
    q_proj_output_int8.min(),
    q_proj_output_int8.max(),
)

print(
    "\nFirst input vector:",
    q_proj_input_int8[0, 0, :16],
)

print(
    "\nFirst weight row:",
    q_proj_weight_int8[0, :16],
)

print(
    "\nFirst output values:",
    q_proj_output_int8[0, 0, :16],
)